# qdmpy_core Fitting Tutorial

This tutorial covers the `FitManager` and `ConstraintManager` in depth:

1. Creating a `FitManager` with a named model
2. Inspecting and modifying parameter constraints
3. Running `fit()` and exploring the `FitResult`
4. Using the `"auto"` model selector
5. Fitting a real dataset with custom constraints

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from qdmpy_core.odmr import ODMRData, ODMR
from qdmpy_core.odmr.io import MatlabLoader
from qdmpy_core.odmr.processors import BinningProcessor, NormalizationProcessor
from qdmpy_core.fitting import FitManager, ConstraintManager, CONSTRAINT_TYPES, ModelRegistry

## 1. Load and process data

In [ ]:
DATA_FOLDER = Path.home() / "git" / "qdmpy_core" / "tests" / "data" / "MIL2_FOV1"

loader = MatlabLoader(data_folder=str(DATA_FOLDER))
odmr_data = ODMRData.from_loader(loader)

odmr = ODMR(odmr_data)
odmr.processor_manager.add_processor(BinningProcessor(bin_factor=4))
odmr.processor_manager.add_processor(NormalizationProcessor(method="max"))
odmr.process_data()

proc = odmr.processed_data
proc_da = proc.data
freq_ghz = proc_da.coords["freq_ghz"].values

print(f"Data shape: {proc_da.shape}")
print(f"Freq shape: {freq_ghz.shape}")

## 2. FitManager — configuration

`FitManager` is **stateless**: construct it with a model name and optional
constraint overrides; pass data per call via `fit()`.

```python
FitManager(model_name, constraints=None, *, settings=None, gpu_available=None)
```

Supported model names: `"ESR14N"`, `"ESR15N"`, `"ESRSINGLE"`, `"auto"`.

In [ ]:
fitm = FitManager("ESR14N")
print(fitm)
print(f"Model: {fitm.model_name}")
print(f"Parameter names: {fitm.parameter_names}")

## 3. ConstraintManager

Each parameter can be constrained with lower/upper bounds and a constraint type
(`FREE`, `LOWER`, `UPPER`, `LOWER_UPPER`).  The defaults come from `qdmpy_coreSettings`.

Use `fitm.set_constraints(param, vmin=..., vmax=..., constraint_type=...)` to override.

In [ ]:
# Inspect the model parameters and their units
model = fitm.model
print(f"Parameter names: {model.parameter_names}")
print(f"Units:           {model.units}")

In [ ]:
# Tighter center frequency bound (GHz — all internal values stay in GHz)
fitm.set_constraints("center", vmin=2.7, vmax=3.0, constraint_type="LOWER_UPPER")

# Positive contrast for all contrast_0/1/2 slots (pass "contrast" to set all at once)
fitm.set_constraints("contrast", vmin=0.0, constraint_type="LOWER")

# Minimum linewidth to avoid unphysically narrow dips
fitm.set_constraints("width", vmin=0.001, constraint_type="LOWER")

print("Constraints updated. Parameter names:", fitm.parameter_names)

## 4. Running the fit

```python
res = fitm.fit(data, frequencies, *, pixel_spacing=1.0)
```

`data` — `xr.DataArray` with dims `(polarity, freq_range, y, x, freq_idx)`  
`frequencies` — `(n_frange, n_freq)` array in GHz  
`pixel_spacing` — physical pixel size in metres (default 1.0)

In [ ]:
res = fitm.fit(proc_da, freq_ghz, pixel_spacing=4e-6)

metrics = res.get_fit_quality_metrics()
print(res)
print(f"\nParameters available: {list(res.parameters.keys())}")
print(f"Convergence: {metrics['convergence_rate']:.3%}")
print(f"Mean chi²:   {metrics['mean_chi2']:.2e}")

## 5. Exploring FitResult

`FitResult` provides:
- `res.centers`, `res.linewidths`, `res.contrasts`, `res.offsets` — property shorthands
- `res.parameters[name]` — raw array of shape `(n_pol, n_frange, n_pixel)`
- `res.get_parameter_map(name)` — requires a 1-D slice first
- `res.b111_remanent`, `res.b111_induced` — B₁₁₁ in µT

In [ ]:
# Center frequency map — neg polarity, low range
center_map = res.parameters["center"][0, 0].reshape(res.scan_dimensions)
contrast_map = res.parameters["contrast"][0, 0].reshape(res.scan_dimensions)
chi2_map = res.parameters["chi2"][0, 0].reshape(res.scan_dimensions)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].imshow(center_map, cmap="plasma")
axes[0].set_title("Center (GHz) [neg, low]")
fig.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(contrast_map, cmap="viridis")
axes[1].set_title("Contrast (a.u.) [neg, low]")
fig.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(chi2_map, cmap="hot_r",
                      vmax=np.percentile(chi2_map, 99))
axes[2].set_title("χ² (fit quality) [neg, low]")
fig.colorbar(im2, ax=axes[2], shrink=0.8)

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.tight_layout()
plt.show()

## 6. B₁₁₁ magnetic field maps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vmax = np.percentile(np.abs(res.b111_remanent), 99)
im0 = axes[0].imshow(res.b111_remanent, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0].set_title("B₁₁₁ remanent (µT)")
fig.colorbar(im0, ax=axes[0], shrink=0.8, label="µT")

vmax = np.percentile(np.abs(res.b111_induced), 99)
im1 = axes[1].imshow(res.b111_induced, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[1].set_title("B₁₁₁ induced (µT)")
fig.colorbar(im1, ax=axes[1], shrink=0.8, label="µT")

plt.suptitle("B₁₁₁ components", y=1.01)
plt.tight_layout()
plt.show()

## 7. Auto model selection

`FitManager("auto")` guesses the model from the data on the first call to `fit()`.
It inspects the averaged spectrum and picks `ESR14N`, `ESR15N`, or `ESRSINGLE`
based on the number of visible dips.

In [ ]:
fitm_auto = FitManager("auto")
print(f"Before fit: model = {fitm_auto.model_name}")

res_auto = fitm_auto.fit(proc_da, freq_ghz)
print(f"After fit:  model = {fitm_auto.model_name}")

## 8. Reusing a FitManager

Because `FitManager` is stateless, the same instance can be called repeatedly
with different data (e.g. different bin factors or datasets).

In [ ]:
# Fit binned-by-2 data with the same FitManager
odmr2 = ODMR(odmr_data)
odmr2.processor_manager.add_processor(BinningProcessor(bin_factor=2))
odmr2.processor_manager.add_processor(NormalizationProcessor(method="max"))
odmr2.process_data()
proc_da2 = odmr2.processed_data.data
freq_ghz2 = proc_da2.coords["freq_ghz"].values

res2 = fitm.fit(proc_da2, freq_ghz2)  # same FitManager, different data
metrics2 = res2.get_fit_quality_metrics()
print(f"Bin-2 convergence: {metrics2['convergence_rate']:.3%}")
print(f"Bin-2 scan dims:   {res2.scan_dimensions}")

## Summary

| API | Description |
|-----|------------|
| `FitManager(model_name)` | Construct with `"ESR14N"`, `"ESR15N"`, `"ESRSINGLE"`, or `"auto"` |
| `fitm.set_constraints(param, ...)` | Override default bounds/type for a parameter |
| `fitm.fit(data, freq_ghz)` | Run GPU fit, return `FitResult` |
| `res.parameters[name]` | Raw array `(n_pol, n_frange, n_pixel)` |
| `res.b111_remanent` / `.b111_induced` | 2-D B₁₁₁ field maps in µT |
| `res.get_fit_quality_metrics()` | Dict with chi², convergence rate, timing |

Constraint types: `FREE`, `LOWER`, `UPPER`, `LOWER_UPPER` (from `CONSTRAINT_TYPES` list).